# Day 5 — Classes, Dataclasses & Type Hints (29 Oct)

*Rebuilt from the master Google Doc session (lesson text and every code example). Code cells are not executed: several need Postgres, a running server or files from the original starter zip.*

## Today at a glance

- Goal: stop passing loose dicts around. Learn to bundle data and the code that works on it into your own types (classes), get most of that for free with @dataclass, and describe every value with type hints so tools can catch mistakes before a customer does.

- Deliverable: day05/support_ticket.py: a validated SupportTicket dataclass and a TicketQueue class that load a messy helpdesk export and report SLA breaches. Plus day05/dsa.py.

- Where this sits: Week 1, Phase 1 (Foundation), day 5 of 7. FastAPI (Week 3), Pydantic models, the Claude SDK's response objects and tool schemas (Week 5) are all built on exactly these three ideas.

| Block | Time | What you do |
|---|---|---|
| Theory | 1.5h | Concepts 1 to 5, running every example yourself |
| Hands-on | 2.5h | support_ticket.py, all acceptance criteria |
| DSA | 0.5h | MinStack, group anagrams |
| Revision | 0.5h | Quiz, Remember cells in the notebook, Day 4 open items |

Open items first: in the repo, Day 4's contact_store.py and main.py are still the starter TODOs. If they are not finished on your machine, give them the first 60 minutes of hands-on time. Today's task uses the same validate, raise, collect-the-rejections pattern, so finishing Day 4 makes today faster.

## Recap (2 minutes, from memory)

- What does the recap code below print, and why does False never appear?

- You save state with open(path, "w") and the program crashes before json.dump finishes. What is in the file now, and which Day 4 pattern prevents this?

- What is the difference between python text_utils.py and import text_utils as far as if __name__ == "__main__": is concerned?

In [ ]:
# Recap: predict the output before running.
import json

data = json.loads('{"id": 7, "tags": ["sso"], "vip": false}')
try:
    print(data["vip"], data["owner"])
except KeyError as err:
    print("missing:", err)
else:
    print("all found")
finally:
    print("done")

Answers are on the last page of today's session.

## Concept 1: Classes and objects

### Why it matters for an FDE

Every SDK you will use is made of classes. anthropic.Anthropic() creates a client object that holds your API key and connection settings; client.messages.create(...) is a method on it; the reply is a Message object with .content and .usage. To read those SDKs, and to write your own client for a customer's system, you need to know what a class is and what self means.

### How it works

A class is a blueprint; an object (or instance) is one thing built from it. You know this from JavaScript: class Ticket { constructor() { this.id = ... } }. In Python the constructor is called __init__, and this is called self. The big difference: Python makes self an explicit first parameter of every method. When you call t1.describe(), Python quietly turns it into Ticket.describe(t1).

In [ ]:
# A class is a blueprint; each object made from it holds its own data.
class Ticket:
    # __init__ runs once per new object, like constructor() in a JS class
    def __init__(self, ticket_id, subject, priority="medium"):
        # self is the object being built (JS "this", but written out)
        self.ticket_id = ticket_id  # an attribute: data stored ON the object
        self.subject = subject
        self.priority = priority
        self.comments = []  # a NEW list for every ticket (safe)

    # a method is a function that lives on the class and gets self first
    def add_comment(self, text):
        self.comments.append(text)
        return len(self.comments)  # return a value, do not just print

    def describe(self):
        return f"{self.ticket_id} [{self.priority}] {self.subject}"


t1 = Ticket("T-101", "Login fails on SSO")  # no "new" keyword in Python
t2 = Ticket("T-102", "Export is slow", priority="high")
print(t1.describe())  # -> T-101 [medium] Login fails on SSO
print(t2.describe())  # -> T-102 [high] Export is slow
print(t1.add_comment("Asked for a HAR file"))  # -> 1
print(len(t1.comments), len(t2.comments))  # -> 1 0  (each has its own list)
# Python passes t1 as self for you, so these two calls are the same
print(Ticket.describe(t1) == t1.describe())  # -> True
# REMEMBER: self.x = ... inside __init__ is what gives each object its data.

Data stored on the object (self.subject) is an attribute. A function defined inside the class is a method. Methods that start and end with double underscores are "dunder" methods: Python calls them for you. __init__ runs on creation, __repr__ decides how the object prints, and __eq__ decides what == means.

In [ ]:
# __repr__ controls how an object prints; __eq__ controls what == means.
class Plain:
    def __init__(self, ticket_id):
        self.ticket_id = ticket_id


# without __repr__ you only see the class name and a memory address
print(Plain("T-1"))  # -> <__main__.Plain object at 0x...>  (address varies)


class Ticket:
    def __init__(self, ticket_id, status="open"):
        self.ticket_id = ticket_id
        self.status = status

    # "dunder" = double underscore. Python calls these for you.
    # __repr__: how the object looks in print() and logs (like toString())
    def __repr__(self):
        return f"Ticket({self.ticket_id!r}, status={self.status!r})"

    # __eq__: without it, == only asks "is this the very same object?"
    def __eq__(self, other):
        if not isinstance(other, Ticket):
            return NotImplemented  # let Python handle other types
        return self.ticket_id == other.ticket_id


a = Ticket("T-7")
b = Ticket("T-7", status="closed")
print(a)  # -> Ticket('T-7', status='open')
print(a == b, a is b)  # -> True False  (same id, but two objects)

### Real-world FDE example

A customer's ticketing API returns tickets as JSON. Instead of passing dicts through 15 functions and hoping every one spells "customer_email" the same way, you define a Ticket class once. In Contentstack terms: a content type is the class (it defines the fields), and each entry is an instance with its own values.

### Common mistakes

- Forgetting self as the first parameter. Python then complains that the method takes 0 positional arguments but 1 was given: the 1 is the object itself.

- Writing subject = subject inside __init__ instead of self.subject = subject. That only creates a local variable that disappears when __init__ ends.

- Calling a method without parentheses: lamp.toggle just looks the method up, lamp.toggle() runs it.

- Expecting print(obj) to show the data. Without __repr__ you get <__main__.Ticket object at 0x...>.

## Concept 2: Class attributes, classmethods and properties

### Why it matters for an FDE

Three small tools make classes practical for API data: values shared by every object (allowed priorities), a second way to build an object (from a raw JSON dict), and values computed from the data instead of stored (is this ticket urgent?). You will see all three in every SDK and write them in today's task.

### How it works

A variable written directly in the class body is a class attribute: one value shared by all objects, like a static field in JS. @classmethod makes a method receive the class itself (cls) instead of an object, which is ideal for alternate constructors such as from_dict(). @property turns a method into something you read like an attribute, without parentheses.

In [ ]:
# Class attributes, an alternate constructor, and a computed property.
class Ticket:
    # class attribute: ONE value shared by every ticket (like a static field)
    VALID_PRIORITIES = ("low", "medium", "high", "urgent")

    def __init__(self, ticket_id, subject, priority="medium"):
        if priority not in Ticket.VALID_PRIORITIES:
            raise ValueError(f"unknown priority {priority!r}")
        self.ticket_id = ticket_id
        self.subject = subject
        self.priority = priority

    # @classmethod receives the CLASS (cls) instead of an object (self).
    # FDE TIP: from_dict() is how API clients turn raw JSON into objects,
    # like a static fromJSON() method in JS.
    @classmethod
    def from_dict(cls, raw):
        return cls(raw["id"], raw["subject"].strip(),
                   raw.get("priority", "medium"))

    # @property: a method you READ like an attribute (no parentheses)
    @property
    def is_urgent(self):
        return self.priority == "urgent"


api_row = {"id": "T-201", "subject": "  SSO loop ", "priority": "urgent"}
t = Ticket.from_dict(api_row)
print(t.subject, t.is_urgent)  # -> SSO loop True
print(t.VALID_PRIORITIES[0])  # -> low  (looked up on the class)
try:
    Ticket("T-202", "Typo in footer", priority="critical")
except ValueError as err:
    print(err)  # -> unknown priority 'critical'

The trap: a class attribute that is mutable (a list, dict or set) is shared by every object. Change it through one object and every other object sees the change.

In [ ]:
# GOTCHA: a mutable CLASS attribute is shared by every object.
class BadTicket:
    tags = []  # ONE list, created once, when the class is defined

    def __init__(self, ticket_id):
        self.ticket_id = ticket_id


class GoodTicket:
    def __init__(self, ticket_id):
        self.ticket_id = ticket_id
        self.tags = []  # a new list for each object, made in __init__


a, b = BadTicket("T-1"), BadTicket("T-2")
a.tags.append("vip")
print(b.tags)  # -> ['vip']  (b sees a's tag: it is the same list)

c, d = GoodTicket("T-3"), GoodTicket("T-4")
c.tags.append("vip")
print(d.tags)  # -> []
# REMEMBER: this is Day 2's mutable-default trap again. Mutable data
# (lists, dicts, sets) belongs in __init__, not on the class.

### Real-world FDE example

A CMS client class might define BASE_URL as a class attribute, Entry.from_api(json_body) as a classmethod that every endpoint reuses, and entry.is_published as a property computed from the publish details. A property can never go stale, because it is recalculated every time you read it.

### Common mistakes

- Lists or dicts as class attributes when each object needs its own. Create them in __init__.

- Using self in a classmethod. It receives cls; there is no object yet.

- Calling a property with parentheses: t.is_urgent() fails with 'bool' object is not callable, because t.is_urgent is already the value.

- Hiding slow work (an API call) inside a property. Readers expect a property to be instant.

## Concept 3: Inheritance, in small doses

### Why it matters for an FDE

You will meet inheritance mostly in two places: exception hierarchies (the Anthropic SDK's RateLimitError is a kind of APIStatusError, which is a kind of APIError) and framework base classes you extend. You need to read it fluently; you will write deep hierarchies rarely.

### How it works

class EmailNotifier(Notifier): means EmailNotifier is a Notifier: it gets all of the parent's methods and can override some. super() reaches the parent's version, for example to run the parent's __init__ first. isinstance(obj, Parent) is True for children too.

In [ ]:
# Inheritance: a child class reuses and extends a parent class.
class Notifier:
    def __init__(self, channel):
        self.channel = channel

    def send(self, message):
        return f"[{self.channel}] {message}"


# EmailNotifier "is a" Notifier, like "class B extends A" in JS
class EmailNotifier(Notifier):
    def __init__(self, address):
        super().__init__("email")  # run the parent's __init__ first
        self.address = address

    def send(self, message):  # override: replace the parent's version...
        base = super().send(message)  # ...but still reuse it
        return f"{base} -> {self.address}"


for n in [Notifier("slack"), EmailNotifier("ops@acme.in")]:
    print(n.send("T-101 breached SLA"))  # same call, different behaviour
# -> [slack] T-101 breached SLA
# -> [email] T-101 breached SLA -> ops@acme.in

print(isinstance(EmailNotifier("x@y.in"), Notifier))  # -> True

Now Day 4's custom exceptions make sense. class InvalidContactError(ValueError): was a one-line child class, and except matches the class you name or any of its children.

In [ ]:
# Day 4's custom exceptions, explained: they are tiny child classes.
class TicketError(Exception):  # the base error for our whole module
    """Anything that goes wrong with tickets."""


class InvalidTicketError(TicketError):  # "is a" TicketError
    """One ticket record is bad."""


try:
    raise InvalidTicketError("record 4: subject is empty")
except TicketError as err:  # catching the PARENT also catches the child
    print(type(err).__name__, "|", err)
# -> InvalidTicketError | record 4: subject is empty

print(issubclass(InvalidTicketError, Exception))  # -> True
# FDE TIP: one base error per module lets callers write ONE except clause.

### Real-world FDE example

When you call Claude in production, you catch anthropic.RateLimitError to wait and retry, and let anthropic.AuthenticationError fail loudly because retrying a bad key is pointless. Both inherit from anthropic.APIError, so one except anthropic.APIError at the top of a job can still log anything the SDK raises.

### Common mistakes

- Forgetting super().__init__(...) in a child's __init__, so the parent's attributes never get set.

- Deep inheritance trees for data. Prefer composition: a ticket has a customer (an attribute), it is not a kind of customer.

- Catching the parent class when you only meant the child, which hides errors you did not plan for.

## Concept 4: Dataclasses

### Why it matters for an FDE

Most of the classes you write in integration work are records: a ticket, a CMS entry, a config. Writing __init__, __repr__ and __eq__ by hand for each one is slow and easy to get wrong. @dataclass generates them from a list of typed fields. Pydantic, which FastAPI and the Claude SDK use, follows the same idea and adds runtime validation.

### How it works

Put @dataclass above a class and list the fields with type hints. Python writes the boring methods for you. Fields with defaults must come after fields without. For a list or dict default use field(default_factory=list), so every object gets its own new list.

In [ ]:
# @dataclass writes __init__, __repr__ and __eq__ for you.
from dataclasses import dataclass, field


@dataclass
class Ticket:
    # each line is a FIELD: a name, a type hint, and optionally a default
    ticket_id: str
    subject: str
    priority: str = "medium"  # fields with defaults must come last
    # GOTCHA: "tags: list[str] = []" is refused. default_factory=list
    # calls list() for EACH new ticket, so no list is ever shared.
    tags: list[str] = field(default_factory=list)


t = Ticket("T-1", "Webhook")
print(t)  # the generated __repr__ shows every field
# -> Ticket(ticket_id='T-1', subject='Webhook', priority='medium', tags=[])
same = Ticket("T-1", "Webhook")
print(t == same, t is same)  # -> True False  (== compares field values)
t.tags.append("api")  # dataclass objects are normal, changeable objects
print(t.tags, same.tags)  # -> ['api'] []

__post_init__ runs right after the generated __init__: the place to clean and validate. asdict() converts the object, including nested dataclasses, into plain dicts and lists ready for json.dumps().

In [ ]:
# __post_init__ cleans and validates; asdict() makes JSON-ready data.
import json
from dataclasses import asdict, dataclass, field


@dataclass
class Ticket:
    ticket_id: str
    subject: str
    tags: list[str] = field(default_factory=list)

    # runs right after the generated __init__: the place for validation
    def __post_init__(self):
        self.subject = self.subject.strip()
        if not self.subject:
            raise ValueError(f"{self.ticket_id}: subject is empty")
        # normalise tags: lowercase, no duplicates, sorted (Day 3 set idea)
        self.tags = sorted({tag.lower() for tag in self.tags})


t = Ticket("T-2", "  Slow CSV ", ["CSV", "csv", "Export"])
print(t.subject, t.tags)  # -> Slow CSV ['csv', 'export']
print(json.dumps(asdict(t)))  # asdict: dataclass -> plain dict (nested too)
# -> {"ticket_id": "T-2", "subject": "Slow CSV", "tags": ["csv", "export"]}

try:
    Ticket("T-3", "   ")
except ValueError as err:
    print(err)  # -> T-3: subject is empty

@dataclass(frozen=True) makes objects read-only. Use it for values that should never change after creation, such as an SLA rule. To "change" one, make a modified copy with replace().

In [ ]:
# frozen=True makes objects read-only: safe to share, usable as dict keys.
from dataclasses import FrozenInstanceError, dataclass, replace


@dataclass(frozen=True)
class Sla:
    priority: str
    hours: int


urgent = Sla("urgent", 4)
try:
    urgent.hours = 8  # not allowed on a frozen dataclass
except FrozenInstanceError as err:
    print("refused:", err)  # -> refused: cannot assign to field 'hours'

# replace() makes a changed COPY, like {...sla, hours: 8} in JS
relaxed = replace(urgent, hours=8)
print(urgent.hours, relaxed.hours)  # -> 4 8
# frozen + eq means Python can hash it, so it works as a dict key
print({urgent: "page on-call"}[Sla("urgent", 4)])  # -> page on-call

| Feature | dict | plain class | dataclass |
|---|---|---|---|
| Boilerplate | none | __init__, __repr__, __eq__ by hand | one decorator |
| print() | readable | <object at 0x...> unless you add __repr__ | readable, all fields |
| == compares | values | identity, unless you add __eq__ | field values |
| Misspelled field | KeyError, or None with .get() | AttributeError | AttributeError, and mypy warns first |
| Best for | raw JSON in and out | behaviour plus internal state (API clients, queues) | records (tickets, entries, configs) |

### Real-world FDE example

For a helpdesk migration you load 40,000 tickets. As dataclasses, a failing test prints Ticket(id='T-9', status='open', ...) instead of a memory address, two exports can be compared with ==, and the cleaning rules live in one __post_init__ instead of being repeated in every script.

### Common mistakes

- tags: list[str] = []. Python refuses it with "mutable default ... is not allowed: use default_factory".

- A field without a default after one with a default: a TypeError when the class is defined.

- Thinking the type hints are checked. Ticket(42, None) is accepted; validate in __post_init__ (or use Pydantic later).

- Using asdict() on data containing a datetime and expecting json.dumps() to work. Convert dates with .isoformat() first.

## Concept 5: Type hints and enums

### Why it matters for an FDE

Type hints are how modern Python documents itself. FastAPI reads them to validate requests and generate API docs, Pydantic reads them to parse JSON, and tool definitions for Claude are JSON Schemas that libraries can generate from typed models. In a customer handover, typed code is code the client's team can maintain without you.

### How it works

A hint follows a colon for parameters and variables, and an arrow for the return value. Python stores hints but does not enforce them at runtime. That is the part that surprises people coming from TypeScript, where the compiler refuses to build.

In [ ]:
# Type hints say what a value SHOULD be. Python does not enforce them.
def sla_hours(priority: str, business_days: bool = False) -> int:
    hours = {"urgent": 4, "high": 24}.get(priority, 72)
    return hours * 2 if business_days else hours


print(sla_hours("urgent"))  # -> 4
print(sla_hours("high", business_days=True))  # -> 48
# GOTCHA: a wrong type still runs. Hints are for tools and humans.
print(sla_hours(42))  # -> 72  (no error: 42 is simply not in the dict)
print(sla_hours.__annotations__["return"])  # -> <class 'int'>

In [ ]:
# The hint shapes you will use most (Python 3.10+ syntax).
from datetime import datetime


def find_ticket(tickets: list[dict], ticket_id: str) -> dict | None:
    """Return the matching ticket, or None (that is what "| None" says)."""
    for t in tickets:
        if t["id"] == ticket_id:
            return t
    return None


def count_by_status(tickets: list[dict]) -> dict[str, int]:
    counts: dict[str, int] = {}  # you can hint a variable too
    for t in tickets:
        counts[t["status"]] = counts.get(t["status"], 0) + 1
    return counts


rows = [{"id": "T-1", "status": "open"}, {"id": "T-2", "status": "open"},
        {"id": "T-3", "status": "closed"}]
print(find_ticket(rows, "T-3"))  # -> {'id': 'T-3', 'status': 'closed'}
print(find_ticket(rows, "T-9"))  # -> None
print(count_by_status(rows))  # -> {'open': 2, 'closed': 1}

# ISO text from an API -> a real datetime you can compare and subtract
opened: datetime = datetime.fromisoformat("2026-10-29T03:30:00+00:00")
print(opened.hour, opened.tzinfo)  # -> 3 UTC

| Hint | Means | Example value |
|---|---|---|
| int, str, float, bool | that type | 3, "T-1", 2.5, True |
| list[str] | a list of strings | ["sso", "auth"] |
| dict[str, int] | str keys, int values | {"open": 2} |
| tuple[str, int] | exactly two items: a str, then an int | ("the", 3) |
| str \| None | a str or None | the result of a lookup |
| -> None | the function returns nothing | save functions |

To get TypeScript-style checking you run a type checker such as mypy. It reads the hints without running the code and reports mismatches. Save the file below as hints.py and compare the two commands.

In [ ]:
# hints.py: a type checker reads hints WITHOUT running the code.
def total_hours(hours: list[int]) -> int:
    return sum(hours)


print(total_hours([4, 24]))  # -> 28
print(total_hours(["4", "24"]))  # wrong type: crashes only when it runs

```bash
# Terminal: run the file, then ask mypy to check it without running it
$ python hints.py
28
Traceback (most recent call last):
  ...
TypeError: unsupported operand type(s) for +: 'int' and 'str'

$ pip install mypy        # Day 6 covers pip and virtual environments
$ mypy --hide-error-codes hints.py
hints.py:7: error: List item 0 has incompatible type "str"; expected "int"
hints.py:7: error: List item 1 has incompatible type "str"; expected "int"
Found 2 errors in 1 file (checked 1 source file)
```

An enum is a fixed set of named values. StrEnum (Python 3.11+) members are also strings, so they compare equal to the raw text and go into JSON without conversion. Turning API text into an enum validates it in the same step.

In [ ]:
# An Enum is a fixed set of named values: no more typo'd status strings.
from enum import StrEnum  # Python 3.11+, like a TypeScript string enum


class Status(StrEnum):
    OPEN = "open"
    PENDING = "pending"
    CLOSED = "closed"


s = Status("pending")  # raw API text -> enum member (and validates it)
print(s, s is Status.PENDING)  # -> pending True
print(s == "pending")  # -> True  (a StrEnum IS a str, so JSON just works)
print([m.value for m in Status])  # -> ['open', 'pending', 'closed']
try:
    Status("resolved")  # not one of the allowed values
except ValueError as err:
    print(err)  # -> 'resolved' is not a valid Status

Here is everything from today in one model: a dataclass with typed fields, an enum, a datetime, and the from_api / to_api pair that every integration needs. Today's task asks you to build the same shape for tickets, with validation and SLA logic added.

In [ ]:
# Putting it together: a typed model for a Contentstack-style entry.
from dataclasses import dataclass, field
from datetime import datetime
from enum import StrEnum


class Locale(StrEnum):
    EN_IN = "en-in"
    HI_IN = "hi-in"


@dataclass
class Entry:
    uid: str
    title: str
    locale: Locale
    updated_at: datetime
    tags: list[str] = field(default_factory=list)

    # "Entry" is in quotes because the class is not finished yet
    @classmethod
    def from_api(cls, raw: dict) -> "Entry":
        """Raw JSON dict (strings everywhere) -> typed Entry."""
        return cls(
            uid=raw["uid"],
            title=raw["title"].strip(),
            locale=Locale(raw["locale"]),  # ValueError if unknown
            updated_at=datetime.fromisoformat(raw["updated_at"]),
            tags=list(raw.get("tags") or []),  # None or missing -> []
        )

    def to_api(self) -> dict:
        """Typed Entry -> JSON-safe dict (enum and datetime as text)."""
        return {"uid": self.uid, "title": self.title,
                "locale": str(self.locale),
                "updated_at": self.updated_at.isoformat(),
                "tags": self.tags}


raw = {"uid": "blt9", "title": " Pricing ", "locale": "hi-in",
       "updated_at": "2026-10-29T03:30:00+00:00"}
entry = Entry.from_api(raw)
print(entry.locale is Locale.HI_IN, entry.updated_at.year)  # -> True 2026
print(entry.to_api()["title"], entry.to_api()["tags"])  # -> Pricing []
print(Entry.from_api(entry.to_api()) == entry)  # -> True  (round trip)

### Real-world FDE example

A support bot built on Claude must route tickets by priority. If priority is a free string, one record says "Urgent", another "URGENT", another "p1", and your routing silently misses them. With a Priority enum, unknown values fail at import time with a clear message, and your editor autocompletes Priority.URGENT.

### Common mistakes

- Believing a hint protects you at runtime. It does not; validate data that comes from outside.

- Old syntax from tutorials: List[str] and Optional[str] from typing. Still valid, but on Python 3.10+ write list[str] and str | None.

- Comparing enum members with raw strings everywhere. Convert once at the edge (from_dict), then use Status.CLOSED inside your code.

- Using a class name in a hint inside its own body without quotes (-> Entry). On Python 3.13 and older that raises NameError; write -> "Entry".

## Check your understanding (predict first, then run)

Write down your prediction for each numbered part, then run it in the notebook or a scratch file.

In [ ]:
# Predict each output BEFORE you run this cell.
from dataclasses import dataclass

# 1
class Counter:
    hits = []

    def hit(self):
        self.hits.append(1)


x, y = Counter(), Counter()
x.hit()
y.hit()
print(len(x.hits))

# 2
@dataclass
class Tag:
    name: str


print(Tag("sso") == Tag("sso"), Tag("sso") is Tag("sso"))

# 3
def double(n: int) -> int:
    return n * 2


print(double("ab"))

# 4
class Light:
    def __init__(self):
        self.on = False

    def toggle(self):
        self.on = not self.on


lamp = Light()
lamp.toggle
print(lamp.on)

# 5
try:
    @dataclass
    class Bad:
        tags: list = []
except ValueError as err:
    print(type(err).__name__)

## Hands-on task (about 2.5 hours): day05/support_ticket.py

A client's support team exports tickets from their helpdesk as JSON. Before an AI assistant can triage them, you need clean, typed objects. Your code must turn the raw export into SupportTicket objects, reject bad records with clear messages, and answer "which tickets have breached their SLA?".

```bash
# day05/ layout for today's task
day05/
    support_ticket.py      # the module: enums, SupportTicket, TicketQueue
    tickets_export.json    # the messy helpdesk export (given, do not edit)
    dsa.py                 # today's two DSA problems
    day05_notes.ipynb      # runnable examples from this session
```

The starter file gives you the Priority and Status enums, SLA_HOURS, the two exception classes, a fixed NOW, and TicketQueue.__init__ / __iter__. You write:

- The SupportTicket dataclass fields: id, subject, customer_email, created_at: datetime, priority (default MEDIUM), status (default OPEN), tags (default: a new empty list).

- __post_init__: strip the subject (empty is an error), strip and lowercase the email (no "@" is an error), and lowercase, dedupe and sort the tags. Raise InvalidTicketError with the messages shown in the starter.

- from_dict(cls, raw): a classmethod that turns one raw record into a ticket. Missing priority or status means MEDIUM or OPEN; missing or null tags means []; an unknown priority raises InvalidTicketError.

- to_dict(), age_hours(now) and is_overdue(now): JSON-safe output, whole hours since creation, and "not closed and older than its SLA".

- TicketQueue.add, __len__, by_status, overdue (oldest first) and summary, then load_queue(path), which never stops on a bad record.

```bash
# What a correct run looks like (run from the repo root)
$ python day05/support_ticket.py
rejected: record 3: subject is empty
rejected: record 4: unknown priority 'critical'
rejected: record 6: email 'john.doe-at-contoso' has no '@'
rejected: record 8: duplicate id 'T-1002'
loaded: 4, rejected: 4
by status: {'open': 2, 'pending': 1, 'closed': 1}
overdue: T-1002 (high, 50h), T-1001 (urgent, 10h)
round trip ok: True
```

Acceptance criteria:

- python day05/support_ticket.py prints exactly the eight lines shown above.

- SupportTicket is a @dataclass; tags uses field(default_factory=list); priority and status are Priority / Status members, never raw strings.

- Building a ticket directly with bad data raises InvalidTicketError, for example SupportTicket("T-1", "   ", "a@b.in", NOW).

- json.dumps(t.to_dict()) works and SupportTicket.from_dict(t.to_dict()) == t for every ticket.

- Every function and method has type hints, and python -c "import support_ticket" run inside day05/ prints nothing.

- Stretch: pip install mypy, then mypy support_ticket.py reports no errors. Skip it if pip is new to you; Day 6 sets this up properly.

- Committed and pushed with a meaningful message.

Hints (no solutions, try first):

- Declare the fields first and test them in a scratch file with print(SupportTicket("T-9", " Slow ", " A@B.in ", NOW)) before writing any method.

- Concept 5's Entry.from_api is the shape of from_dict; the new part is converting ValueError from Priority(...) into InvalidTicketError with raise ... from err.

- now - self.created_at gives a timedelta; look at .total_seconds().

- load_queue is Day 4's import_contacts again: enumerate(records, start=1) and one try around both from_dict() and queue.add().

- For summary, loop over Status itself so statuses with zero tickets still appear.

## DSA practice (30 min)

- MinStack. A class with push(x), pop(), top(), get_min() and __len__, every one O(1), so get_min() cannot loop. Example: push 5, 2, 7, then get_min() is 2; pop twice and get_min() is 5. Hint: keep a second list holding "the minimum so far" at each height.

- Group anagrams. group_anagrams(words: list[str]) -> list[list[str]] groups words with the same letters, in first-seen order. ["eat", "tea", "tan", "ate", "nat", "bat"] gives [["eat", "tea", "ate"], ["tan", "nat"], ["bat"]]. Hint: "".join(sorted(word)) is the same for all anagrams; use it as a dict key with Day 3's setdefault.

## Quiz (3 quick questions)

- What is self, and why does Python make you write it when JavaScript's this is implicit?

- For data coming back from a customer's API, when would you choose a dataclass over a plain dict, and when a plain class over a dataclass?

- You write def total(prices: list[float]) -> float and call total(["1", "2"]). What happens when it runs, and what would have caught it before running?

## What you achieved today

You can define classes with __init__, attributes and methods, control printing and equality with dunder methods, use class attributes, classmethods and properties, read an inheritance hierarchy (and now understand Day 4's custom exceptions), model records with dataclasses including defaults, validation and frozen objects, and describe your code with type hints and enums that a checker like mypy can verify. That is how every SDK and API client you will use from Week 5 is built.

Tomorrow (Day 6): virtual environments, pip and the requests library. You will call a real public API and save the results to CSV, your first program that talks to another system over the network.

## When you're done

Paste into the Forward Deployment Engineer project chat as "Day 5 review": your recap answers, your predictions next to the real outputs, support_ticket.py, the terminal output of the run, both DSA solutions and the quiz answers. Include Day 4's contact_store.py and main.py if they are done. Tracebacks are welcome; send them with what you tried.

## Recap answers

*Your answers:*
